In [ ]:
import pandas as pd
import re
from database import get_connection
from pathlib import Path
from collections import Counter

conn = get_connection()

def query_df(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=columns)

In [ ]:
posts_df = query_df(conn, "SELECT * FROM posts")
comments_df = query_df(conn, "SELECT * FROM comments")

posts_df.info()
print("\n")
comments_df.info()

In [ ]:
# Post cleaning objects
remove_post_columns = ['location_lat', 'location_long', 'location_name', 'category', 'num_reports']
deleted_or_removed = ['[deleted]', '[removed]']

clean_posts_df = posts_df.drop(columns=remove_post_columns) # removing empty columns
clean_posts_df['selftext'] = clean_posts_df['selftext'].replace(deleted_or_removed, "") # preserving documnents with titles but no body text
clean_posts_df['full_text'] = clean_posts_df['title'] + " " + clean_posts_df['selftext'] # combining title and selftext as one field

In [ ]:
# Comment cleaning objects
remove_comment_columns = ['downs']
moderator_authors = ['AutoModerator', 'EntireAd9022', 'OTTO-M0D', 'Ti_xiv', 'W0NT0NY', '[deleted]', 'LoveAndDeepspace-ModTeam']

clean_comments_df = comments_df.drop(columns=remove_comment_columns)
clean_comments_df = clean_comments_df[clean_comments_df['author'].isin(moderator_authors) == False] # Remove moderator comments
clean_comments_df = clean_comments_df[~clean_comments_df['body'].isin(deleted_or_removed)] # Remove deleted and removed comments

In [ ]:
# Cleaning author_user_flair field
flair_counts = query_df(conn, """
    SELECT author_flair_text, COUNT(*) AS n
    FROM (
        SELECT author_flair_text FROM posts WHERE author_flair_text IS NOT NULL
        UNION ALL
        SELECT author_flair_text FROM comments WHERE author_flair_text IS NOT NULL
    ) combined
    GROUP BY author_flair_text
    ORDER BY n DESC
""")

flair_counts

In [ ]:
# Cleaning author_user_flair field
code_counts = Counter()
for _, row in flair_counts.iterrows():
    codes = re.findall(r':([^:]+):', row['author_flair_text'].lower())
    for code in codes:
        code_counts[code] += row['n']

fancount_per_flair_emote = pd.DataFrame(
    [(code, count) for code, count in code_counts.items()],
    columns=['code', 'count']
).sort_values('count', ascending=False)

fancount_per_flair_emote

In [ ]:
# Cleaning author_user_flair field
flair_to_main = {
    'Xavier': ['xavier', 'galaxy-kid-wow', 'GK1', 'galaxy-kid-love', ],
    'Zayne': ['zayne', 'happy-snowman-sigh', 'happy-snowman-drink', 'HS1'],
    'Rafayel': ['rafayel', 'artsy-birb-love', 'artsy-birb-proud', 'artsybirbgenius'],
    'Sylus': ['sylus', 'crow-feeling-good', 'crow-mine-now'],
    'Caleb': ['caleb', 'apple-heart', 'apple', 'apple-yougotthis', 'apple-ha-ha'],
    'Valko' : ['valko']
    #springwhispers is neutral
}

# This dictionary mapping is easier to read and update.
print("Character to Emoji Dict: ", flair_to_main)

# Lowercased keys here so lookups work regardless of the flair text's original casing
emoji_to_character = {
    code.lower(): character
    for character, codes in flair_to_main.items()
    for code in codes
}

# This dictionary mapping is optimized for lookups, since the keys are the actual flair text codes (lowercased) and the values are the character names.
print("Emoji to Character Dict: ", emoji_to_character)

def extract_characters(flair_text, mapping):
    if pd.isna(flair_text):
        return []
    flair_text = flair_text.lower()
    colon_codes = re.findall(r':([^:]+):', flair_text)
    # catches plain-text name mentions too, e.g. "Caleb's Co-Pilot" with no :colons: at all
    plain_matches = [code for code in mapping if code in flair_text]
    all_codes = set(colon_codes) | set(plain_matches)
    characters = {mapping[code] for code in all_codes if code in mapping}
    return sorted(characters)

clean_posts_df['mains'] = clean_posts_df['author_flair_text'].apply(lambda x: extract_characters(x, emoji_to_character))
clean_comments_df['mains'] = clean_comments_df['author_flair_text'].apply(lambda x: extract_characters(x, emoji_to_character))

In [ ]:
output_dir = Path('../analysis/data')
output_dir.mkdir(parents=True, exist_ok=True)

clean_posts_df.to_parquet(output_dir / 'clean_posts.parquet')
clean_comments_df.to_parquet(output_dir / 'clean_comments.parquet')
conn.close()